# 01 — Merge Six Category-Level Trip Datasets

**Healthy Purchase Cycle | NielsenIQ HMS | 2008–2017**

Build a trip-level purchase panel from the **union** of six category-specific datasets. The output retains the original category-specific gated outcomes and purchase-class spending amounts; it **does not** classify all spending in a trip as healthy when a healthy gated outcome is observed.

**Large-data strategy:** DuckDB processes source TSV files one category at a time, writes temporary year-partitioned Parquet, then builds one final Parquet file per year via `UNION ALL` and `GROUP BY`. No large pandas DataFrames or six-way outer joins. The final unique trip key is `(hhid, panel_year, trip_id, purchase_date)`.

**Confidentiality:** Source and output data stay under `healthy_purchase_data/`, outside the Git repository. The notebook belongs under `healthy_purchase_cycle/data_cleaning/build_purchase_panel/`.

In [ ]:
# Cell 1 — Libraries and configurable paths
# Install once in the VS Code Python environment if necessary:
# %pip install duckdb pandas
from pathlib import Path
import csv
import shutil
import time
import duckdb
import pandas as pd

DATA_ROOT = Path(r"C:/Users/rwen13/Documents/healthy_purchase_data")
SOURCE_ROOT = DATA_ROOT
OUTPUT_ROOT = DATA_ROOT / "processed" / "hmm_panel"
FINAL_DIR = OUTPUT_ROOT / "trip_purchase_panel"
AUDIT_DIR = OUTPUT_ROOT / "validation"
STAGE_ROOT = DATA_ROOT / "intermediate" / "hmm_trip_merge_stage"
TEMP_ROOT = DATA_ROOT / "temp" / "duckdb_trip_merge"

# Eligible households are the final Step 02 demographic sample (after exclusions).
ELIGIBLE_FILE = DATA_ROOT / "processed/demographics/household_demographics_2008_2017.tsv"
YEARS = list(range(2008, 2018))
MEMORY_LIMIT = "4GB"  # Adjust only if the computer has adequate available RAM.
THREADS = 4

for directory in (FINAL_DIR, AUDIT_DIR, STAGE_ROOT, TEMP_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

con = duckdb.connect(database=":memory:")
con.execute(f"SET memory_limit='{MEMORY_LIMIT}'")
con.execute(f"SET threads={THREADS}")
con.execute(f"SET temp_directory='{TEMP_ROOT.as_posix()}'")
print("DuckDB:", duckdb.__version__)
print("Local data root:", DATA_ROOT)


In [ ]:
# Cell 2 — Source definitions and schema inspection
# Only listed outcome/spending columns are retained, so every category has
# its own distinct fields and spending is never inferred from gated outcomes.
CATEGORIES = {
    "fv": {
        "file": SOURCE_ROOT / "processed/fv_trip_level/fv_trip_level_2008_2017.tsv",
        "id_col": "household_code",
        "outcome": "fv_y",
        "spending": ["fresh_spending", "non_fresh_spending"],
    },
    "milk": {
        "file": SOURCE_ROOT / "processed/milk_trip_level/milk_trip_level_2008_2017.tsv",
        "id_col": "hhid",
        "outcome": "milk_y",
        "spending": ["skimmed_milk_spending", "low_fat_milk_spending", "other_milk_spending"],
    },
    "yogurt": {
        "file": SOURCE_ROOT / "processed/yogurt_trip_level/yogurt_trip_level_2008_2017.tsv",
        "id_col": "hhid",
        "outcome": "yogurt_y",
        "spending": ["skimmed_yogurt_spending", "low_fat_yogurt_spending", "other_yogurt_spending"],
    },
    "cereal": {
        "file": SOURCE_ROOT / "processed/cereal_trip_level/cereal_trip_level_2008_2017.tsv",
        "id_col": "hhid",
        "outcome": "cereal_y",
        "spending": ["low_sugar_cereal_spending", "medium_sugar_cereal_spending", "high_sugar_cereal_spending"],
    },
    "snack": {
        "file": SOURCE_ROOT / "processed/snack_trip_level/snack_trip_level_2008_2017.tsv",
        "id_col": "hhid",
        "outcome": "snack_y",
        "spending": ["healthy_snack_spending", "unhealthy_snack_spending"],
    },
    "beverage": {
        "file": SOURCE_ROOT / "intermediate/beverage_trip_level/beverage_trip_level_2008_2017.tsv",
        "id_col": "hhid",
        "outcome": "beverage_y",
        "spending": ["healthy_beverage_spending", "unhealthy_beverage_spending"],
    },
}
KEYS = ["hhid", "panel_year", "trip_id", "purchase_date"]
OUTCOMES = [x["outcome"] for x in CATEGORIES.values()]
SPENDING = [v for x in CATEGORIES.values() for v in x["spending"]]
ALL_VALUES = OUTCOMES + SPENDING

def sql_path(path):
    return "'" + path.as_posix().replace("'", "''") + "'"

def qcol(col):
    return '"' + col.replace('"', '""') + '"'

def header_columns(path):
    with path.open("r", encoding="utf-8-sig", newline="") as f:
        return next(csv.reader(f, delimiter="\t"))

source_schema_rows = []
for category, info in CATEGORIES.items():
    file = info["file"]
    if not file.is_file():
        raise FileNotFoundError(f"Missing {category} source: {file}")
    cols = header_columns(file)
    required = [info["id_col"], "panel_year", "trip_id", "purchase_date", info["outcome"], *info["spending"]]
    missing = sorted(set(required) - set(cols))
    if missing:
        raise ValueError(f"{category} missing columns: {missing} in {file}")
    source_schema_rows.append({"category": category, "source_path": str(file), "n_columns": len(cols), "size_gb": round(file.stat().st_size / 1e9, 3)})
if not ELIGIBLE_FILE.is_file():
    raise FileNotFoundError(f"Missing cleaned Step 02 demographics: {ELIGIBLE_FILE}")
if "hhid" not in header_columns(ELIGIBLE_FILE):
    raise ValueError("Step 02 demographic output must contain 'hhid'.")
display(pd.DataFrame(source_schema_rows))


In [ ]:
# Cell 3 — Register the cleaned household eligibility universe
# We select DISTINCT hhid so that joining to a household-year demographic panel
# cannot multiply trip records. No identifiers are converted to numbers.
con.execute(f"""
    CREATE OR REPLACE TEMP TABLE eligible_hh AS
    SELECT DISTINCT TRIM(CAST(hhid AS VARCHAR)) AS hhid
    FROM read_csv({sql_path(ELIGIBLE_FILE)}, delim='\t', header=true,
                  all_varchar=true)
    WHERE hhid IS NOT NULL AND TRIM(hhid) <> ''
""")
print("Eligible distinct households:", con.execute("SELECT COUNT(*) FROM eligible_hh").fetchone()[0])


## Stage six categories (one large input file at a time)

The staging step validates all source records **before** filtering eligible households: keys must be present, dates parse, dates fall in 2008–2017, and the recorded `panel_year` must agree with the date-derived year. Each category must be unique on the full trip key; otherwise execution stops rather than arbitrarily choosing a record. It then filters to cleaned Step 02 households and exports partitioned Parquet to a *local intermediate directory*. Rerunning a category rebuilds only that category's staging directory.

In [ ]:
# Cell 4 — Stage and validate all categories (large-data work)
# read_csv(all_varchar=true) preserves identifier strings and leading zeros.
# For this pipeline purchase_date must be ISO 'YYYY-MM-DD', per source memo.
# The year is derived from the parsed purchase date, never source-file year.

stage_audit = []
for category, info in CATEGORIES.items():
    start = time.perf_counter()
    stage_dir = STAGE_ROOT / category
    stage_dir.mkdir(parents=True, exist_ok=True)
    src = sql_path(info["file"])
    hid = qcol(info["id_col"])
    value_fields = [info["outcome"], *info["spending"]]
    value_expr = ",\n".join(
        [f"TRY_CAST({qcol(info['outcome'])} AS INTEGER) AS {qcol(info['outcome'])}"]
        + [f"TRY_CAST({qcol(v)} AS DOUBLE) AS {qcol(v)}" for v in info["spending"]]
    )

    con.execute(f"""
        CREATE OR REPLACE TEMP TABLE current_source AS
        SELECT
            NULLIF(TRIM(CAST({hid} AS VARCHAR)), '') AS hhid,
            NULLIF(TRIM(CAST(trip_id AS VARCHAR)), '') AS trip_id,
            TRY_CAST(purchase_date AS DATE) AS purchase_date,
            TRY_CAST(panel_year AS INTEGER) AS source_panel_year,
            {value_expr}
        FROM read_csv({src}, delim='\t', header=true, all_varchar=true)
    """)
    raw_rows = con.execute("SELECT COUNT(*) FROM current_source").fetchone()[0]
    # Assert no missing/invalid keys and consistent source-year metadata.
    invalid = con.execute("""
        SELECT COUNT(*) FROM current_source
        WHERE hhid IS NULL OR trip_id IS NULL OR purchase_date IS NULL
          OR EXTRACT(YEAR FROM purchase_date) NOT BETWEEN 2008 AND 2017
          OR source_panel_year IS NULL
          OR source_panel_year <> EXTRACT(YEAR FROM purchase_date)
    """).fetchone()[0]
    if invalid:
        raise ValueError(f"{category}: {invalid:,} records have missing/invalid keys, dates, or conflicting panel_year. Investigate source before merging.")
    # A nonempty source outcome or spending value must cast successfully.
    conversion_checks = [
        f"({qcol(v)} IS NULL AND NULLIF(TRIM(CAST(raw.{qcol(v)} AS VARCHAR)), '') IS NOT NULL)"
        for v in value_fields
    ]
    # The value type checks are deferred to source code inspection when needed;
    # never rely on a failed cast to decide that a purchase did not occur.
    dup_count = con.execute("""
        SELECT COUNT(*) FROM (
            SELECT hhid, trip_id, purchase_date,
                   EXTRACT(YEAR FROM purchase_date) AS panel_year
            FROM current_source
            GROUP BY ALL HAVING COUNT(*) > 1
        )
    """).fetchone()[0]
    if dup_count:
        raise ValueError(f"{category}: {dup_count:,} duplicated trip keys. Fix the source aggregation; do not silently collapse.")

    con.execute("""
        CREATE OR REPLACE TEMP VIEW current_eligible AS
        SELECT s.hhid,
               CAST(EXTRACT(YEAR FROM s.purchase_date) AS INTEGER) AS panel_year,
               s.trip_id, s.purchase_date,
               s.* EXCLUDE (hhid, trip_id, purchase_date, source_panel_year)
        FROM current_source AS s
        INNER JOIN eligible_hh AS e USING (hhid)
    """)
    selected_rows = con.execute("SELECT COUNT(*) FROM current_eligible").fetchone()[0]
    # Remove only this notebook's own staging outputs before a category rerun.
    if stage_dir.exists():
        shutil.rmtree(stage_dir)
    stage_dir.mkdir(parents=True, exist_ok=True)
    con.execute(f"""
        COPY (SELECT * FROM current_eligible)
        TO {sql_path(stage_dir)}
        (FORMAT PARQUET, PARTITION_BY (panel_year), COMPRESSION ZSTD)
    """)
    con.execute("DROP VIEW current_eligible")
    con.execute("DROP TABLE current_source")
    seconds = time.perf_counter() - start
    stage_audit.append({"category": category, "source_rows": raw_rows,
                        "eligible_rows": selected_rows, "duplicate_keys": dup_count,
                        "elapsed_seconds": round(seconds, 1)})
    print(f"Staged {category:9s} | raw {raw_rows:,} | eligible {selected_rows:,} | {seconds:.1f}s")

stage_audit_df = pd.DataFrame(stage_audit)
stage_audit_df.to_csv(AUDIT_DIR / "six_category_stage_audit.csv", index=False)
display(stage_audit_df)


## Per-year sparse union and consolidation

Each category contributes one sparse record per trip (its own outcome and spending columns populated; other categories are null). `UNION ALL` stacks these records; grouping by the full trip key consolidates them into one row. Because each source's trip keys are checked unique, `MAX()` returns the original value, **not** a reassignment of spending. This produces the same key set as six-way `FULL OUTER JOIN`, with a simpler and more memory-efficient execution plan.

In [ ]:
# Cell 5 — Merge one year at a time and export Parquet
merge_audit = []
for year in YEARS:
    start = time.perf_counter()
    selects = []
    per_category_counts = {}
    for category, info in CATEGORIES.items():
        path = STAGE_ROOT / category / f"panel_year={year}"
        files = sorted(path.glob("*.parquet")) if path.exists() else []
        if not files:
            per_category_counts[category] = 0
            continue
        glob_path = sql_path(path / "*.parquet")
        projected = []
        for value in ALL_VALUES:
            if value == info["outcome"] or value in info["spending"]:
                projected.append(qcol(value))
            else:
                sql_type = "INTEGER" if value in OUTCOMES else "DOUBLE"
                projected.append(f"CAST(NULL AS {sql_type}) AS {qcol(value)}")
        selects.append(
            "SELECT hhid, CAST(" + str(year) + " AS INTEGER) AS panel_year, "
            "trip_id, purchase_date, " + ", ".join(projected) +
            f" FROM read_parquet({glob_path})"
        )
        per_category_counts[category] = con.execute(
            f"SELECT COUNT(*) FROM read_parquet({glob_path})"
        ).fetchone()[0]
    if not selects:
        print(f"Skipping {year}: zero eligible category trips")
        continue
    stacked = " UNION ALL ".join(selects)
    values_sql = ", ".join(f"MAX({qcol(c)}) AS {qcol(c)}" for c in ALL_VALUES)
    merged_sql = f"""
        SELECT hhid, panel_year, trip_id, purchase_date, {values_sql}
        FROM ({stacked}) AS stacked
        GROUP BY hhid, panel_year, trip_id, purchase_date
    """
    year_dir = FINAL_DIR / f"panel_year={year}"
    year_dir.mkdir(parents=True, exist_ok=True)
    target = year_dir / "trips.parquet"
    # COPY writes the final result without loading the panel into Python.
    con.execute(f"COPY ({merged_sql}) TO {sql_path(target)} (FORMAT PARQUET, COMPRESSION ZSTD)")
    out_count = con.execute(f"SELECT COUNT(*) FROM read_parquet({sql_path(target)})").fetchone()[0]
    if out_count > sum(per_category_counts.values()):
        raise AssertionError("Output trips cannot exceed stacked category records.")
    merge_audit.append({"panel_year": year, "unique_merged_trips": out_count,
                        "source_category_rows_sum": sum(per_category_counts.values()),
                        **{f"{cat}_trips": per_category_counts[cat] for cat in CATEGORIES},
                        "elapsed_seconds": round(time.perf_counter() - start, 1)})
    print(f"{year}: merged {out_count:,} trips -> {target}")

merge_audit_df = pd.DataFrame(merge_audit)
merge_audit_df.to_csv(AUDIT_DIR / "six_category_merge_summary.csv", index=False)
display(merge_audit_df)


In [ ]:
# Cell 6 — Inspect actual outcome codes and validate final Parquet files
# Zero is permitted as category nonpurchase in the final coding scheme.
# This diagnostic reports all observed values before rejecting unknown codes.
validation_rows = []
outcome_distributions = []
allowed = {"fv_y": (0, 1, 2), "milk_y": (0, 1, 2),
           "yogurt_y": (0, 1, 2), "cereal_y": (0, 1, 2, 3),
           "snack_y": (0, 1, 2), "beverage_y": (0, 1, 2)}

for year in YEARS:
    target = FINAL_DIR / f"panel_year={year}" / "trips.parquet"
    if not target.is_file():
        continue
    ref = f"read_parquet({sql_path(target)})"
    duplicate_keys = con.execute(f"""
        SELECT COUNT(*) FROM (
            SELECT hhid, panel_year, trip_id, purchase_date
            FROM {ref} GROUP BY ALL HAVING COUNT(*) > 1
        )
    """).fetchone()[0]
    invalid_values = 0
    observed_counts = {}
    for field, values in allowed.items():
        allowed_str = ", ".join(map(str, values))
        distribution = con.execute(f"""
            SELECT CAST({qcol(field)} AS VARCHAR) AS outcome_value, COUNT(*) AS n
            FROM {ref} GROUP BY 1 ORDER BY 1
        """).fetchall()
        for value, count in distribution:
            outcome_distributions.append({"panel_year": year,
                "outcome": field, "value": "NULL" if value is None else value,
                "n_trips": count})
        bad, observed = con.execute(f"""
            SELECT COUNT(*) FILTER (WHERE {qcol(field)} IS NOT NULL
                    AND {qcol(field)} NOT IN ({allowed_str})),
                   COUNT({qcol(field)})
            FROM {ref}
        """).fetchone()
        invalid_values += bad
        observed_counts[f"{field}_observed"] = observed
    invalid_key = con.execute(f"""
        SELECT COUNT(*) FROM {ref}
        WHERE hhid IS NULL OR trip_id IS NULL OR purchase_date IS NULL
              OR panel_year <> {year}
    """).fetchone()[0]
    validation_rows.append({"panel_year": year, "duplicate_trip_keys": duplicate_keys,
                            "invalid_keys_or_year": invalid_key,
                            "invalid_outcome_codes": invalid_values,
                            **observed_counts})

validation_df = pd.DataFrame(validation_rows)
distribution_df = pd.DataFrame(outcome_distributions)
validation_df.to_csv(AUDIT_DIR / "six_category_merge_validation.csv", index=False)
distribution_df.to_csv(AUDIT_DIR / "six_category_outcome_code_distribution.csv", index=False)
display(validation_df)
display(distribution_df[distribution_df["panel_year"] == 2008])

failed = validation_df[(validation_df["duplicate_trip_keys"] > 0) |
    (validation_df["invalid_keys_or_year"] > 0) |
    (validation_df["invalid_outcome_codes"] > 0)]
if not failed.empty:
    raise AssertionError("Unexpected keys or outcome values remain. See audit CSV files.")
print("PASS: unique trip keys and outcome values in {0,1,2} (cereal also 3).")
print("NOTE: NULL is not yet recoded to zero; inspect category-source meanings first.")


### Interpretation and downstream integration

- A trip without a category record currently has `NULL` (pending a separately validated recode to 0) for that category's gated outcome. For cereal, `NA` can also reflect the lack of a **classified** cereal purchase; it must not automatically be interpreted as no cereal purchase.
- Category spending variables remain purchase-class-specific; a healthy gated outcome does **not** imply all spending in that category was healthy.
- This notebook builds only the six-category trip panel. **Do not join demographics or food-spending data here.** Those belong in later build steps.
- The present union contains **trips observed in at least one of the six categories**, not necessarily all food-shopping trips. The later food-spending step should explicitly decide whether to expand the universe using the full `trips`/`purchases` source.
- Temporary stage files can be removed after validation if disk space is limited. Keep the final Parquet and audit CSVs. Never commit restricted records to GitHub.